# Case Study Expression Plot

Compare one perturbation across `TriShift`, `Scouter`, `GenePert`, and `GEARS`, similar in purpose to Scouter `Fig2_d`, but using this repo and the current DEG selection logic.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "scripts").exists() and (repo_root.parent / "scripts").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from scripts.trishift.analysis._result_adapter import load_payload_item

sns.set_style("whitegrid")
repo_root


## Parameters


In [ ]:
dataset = "adamson"
condition = "BHLHE40+ctrl"
split_id = 1
models = ["trishift_nearest", "scouter", "genepert", "gears"]
space = "deg"  # deg | full_gene
top_k = 12
rank_by = "truth_abs_delta"  # truth_abs_delta | pred_abs_delta
out_root = repo_root / "notebooks" / "artifacts" / f"case_expression_{dataset}_{condition.replace('+', '_')}"
out_root.mkdir(parents=True, exist_ok=True)
palette = {
    "Truth": "#7f7f7f",
    "trishift_nearest": "#1f77b4",
    "scouter": "#f2b701",
    "genepert": "#2ca02c",
    "gears": "#d62728",
}


In [ ]:
model_frames = []
truth_ref = None
metrics_rows = []

for model_name in models:
    _, payload = load_payload_item(dataset=dataset, model_name=model_name, split_id=split_id, condition=None)
    item = payload[condition]
    if space == "full_gene" and "Pred_full" in item:
        pred = np.asarray(item["Pred_full"], dtype=float)
        truth = np.asarray(item["Truth_full"], dtype=float)
        ctrl = np.asarray(item["Ctrl_full"], dtype=float)
        genes = np.asarray(item["gene_name_full"]).astype(str)
    else:
        pred = np.asarray(item["Pred"], dtype=float)
        truth = np.asarray(item["Truth"], dtype=float)
        ctrl = np.asarray(item["Ctrl"], dtype=float)
        genes = np.asarray(item.get("DE_name", [f"g{i}" for i in range(pred.shape[1])])).astype(str)

    truth_delta = truth.mean(axis=0) - ctrl.mean(axis=0)
    pred_delta = pred.mean(axis=0) - ctrl.mean(axis=0)
    if truth_ref is None:
        truth_ref = pd.DataFrame({"Gene": genes, "truth_delta": truth_delta, "pred_delta_ref": pred_delta})
    metrics_rows.append({
        "model_name": model_name,
        "pearson": float(np.corrcoef(truth_delta, pred_delta)[0, 1]) if np.std(truth_delta) > 0 and np.std(pred_delta) > 0 else float("nan"),
        "mae": float(np.mean(np.abs(truth_delta - pred_delta))),
    })
    model_frames.append(pd.DataFrame({"Gene": genes, "Group": model_name, "Expression": pred_delta}))

truth_plot = truth_ref.copy()
truth_plot["Group"] = "Truth"
truth_plot = truth_plot.rename(columns={"truth_delta": "Expression"})[["Gene", "Group", "Expression"]]
plot_df = pd.concat([truth_plot, *model_frames], ignore_index=True)

rank_col = "truth_delta" if rank_by == "truth_abs_delta" else "pred_delta_ref"
order_df = truth_ref.copy()
order_df["rank_score"] = order_df[rank_col].abs()
gene_order = order_df.sort_values("rank_score", ascending=False)["Gene"].head(top_k).tolist()
plot_df = plot_df[plot_df["Gene"].isin(gene_order)].copy()
plot_df["Gene"] = pd.Categorical(plot_df["Gene"], categories=gene_order, ordered=True)
metrics_df = pd.DataFrame(metrics_rows)
plot_df.to_csv(out_root / "case_expression_values.csv", index=False, encoding="utf-8-sig")
metrics_df.to_csv(out_root / "case_expression_metrics.csv", index=False, encoding="utf-8-sig")
display(metrics_df)
display(plot_df.head(20))


In [ ]:
fig_path = out_root / "case_expression_barplot.png"
plt.figure(figsize=(max(10, 0.85 * top_k), 5.8), dpi=220)
ax = sns.barplot(data=plot_df, x="Gene", y="Expression", hue="Group", palette=palette, estimator="mean", errorbar=None)
for bar in ax.patches:
    bar.set_edgecolor("black")
    bar.set_linewidth(0.5)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_xlabel("")
ax.set_ylabel("Delta expression over control")
ax.set_title(f"Case study: {condition} ({dataset}, split {split_id})")
ax.legend(title="", frameon=False, ncol=min(5, len(plot_df["Group"].unique())))
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(fig_path)
plt.close()
display(Image(filename=str(fig_path), width=1400))
for path in [fig_path, out_root / "case_expression_values.csv", out_root / "case_expression_metrics.csv"]:
    print(path)
